# V2 검증 / test 평가
기존 평가·획별 확률·soft mask·불확실성 시각화를 재사용합니다. 기본 zip_val은 ZIP 학습 때 저장한 CSV/seed로 val 분할을 복원합니다. 독립 test 점수가 아닙니다. 최신 ZIP 학습 실행의 best.pt를 자동 선택하며 checkpoint로 직접 지정할 수도 있습니다. 별도 test_dataset 전체 평가는 MODE="test_csv"로 선택하세요. GPU에서 두 셀을 실행합니다.


In [ ]:
from pathlib import Path
import sys, subprocess, tempfile
root = Path(tempfile.mkdtemp(prefix="reference_v2_", dir="/content"))
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/yechan25/hsqm.git", str(root)], check=True)
if not (root / "requirements.txt").is_file():
    raise FileNotFoundError("저장소에서 requirements.txt를 찾지 못했습니다.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements.txt")], check=True)
sys.path.insert(0, str(root))
for key in list(sys.modules):
    if key == "stroke_model" or key.startswith("stroke_model."):
        del sys.modules[key]
commit = subprocess.check_output(["git", "-C", str(root), "rev-parse", "--short", "HEAD"], text=True).strip()
print("모델 코드 준비 완료 / GitHub 버전:", commit)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from stroke_model.test_reference import run_test, run_zip_validation
MODE = "zip_val"  # "test_csv": 기존 별도 test CSV 전체 평가
hsqm = Path("/content/drive/Shareddrives/2026 자율연구/HSQM")
checkpoint = None  # 특정 best.pt를 평가하려면 전체 경로 지정
if checkpoint is None:
    candidates = sorted((hsqm / "stroke_zip_runs").glob("*/best.pt"))
    if not candidates:
        raise FileNotFoundError("ZIP 학습 best.pt가 없습니다. 학습 완료 후 실행하거나 checkpoint를 지정하세요.")
    checkpoint = candidates[-1]
print("평가 checkpoint:", checkpoint)
test_csv = "/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/test_dataset/paths.csv"
if MODE == "zip_val":
    metrics = run_zip_validation(checkpoint, hsqm / "dataset/automated_train_dataset", preview_count=5)
elif MODE == "test_csv":
    metrics = run_test(checkpoint, test_csv, preview_count=5)
else:
    raise ValueError("MODE는 zip_val 또는 test_csv여야 합니다")
